# Notebook 2: Train Random Forest
This notebook trains a baseline Random Forest model on the ICU dataset in Google Colab.
Since Random Forest doesn't naturally handle 3D sequence data `(N, 24, 37)`, we will flatten the time dimension to `(N, 24*37)`.


In [ ]:
import os
try:
    from google.colab import drive
    # Only mount if the drive isn't already mounted to avoid errors
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    data_dir = '/content/drive/MyDrive/ICU-Patient-Deterioration/preprocessing_pipeline/output'
    print('Running in Google Colab. Data directory set to Google Drive.')
except ImportError:
    # If not in Colab (running locally)
    data_dir = 'preprocessing_pipeline/output'
    print('Running Locally. Data directory set to local folder.')


In [ ]:
import numpy as np
import os

# Load Data
print('Loading data...')
X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))

print(f'Original X_train shape: {X_train.shape}')


### Flatten 3D Sequences for Random Forest
We reshape `(N, Sequence_Length, Features)` into `(N, Sequence_Length * Features)`.


In [ ]:
N_train, seq_len, n_features = X_train.shape
N_val = X_val.shape[0]

X_train_flat = X_train.reshape(N_train, seq_len * n_features)
X_val_flat = X_val.reshape(N_val, seq_len * n_features)

print(f'Flattened X_train shape: {X_train_flat.shape}')
print(f'Flattened X_val shape: {X_val_flat.shape}')


### Train the Random Forest
We will use `n_jobs=-1` to utilize all CPU cores in Colab.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

print('Applying SMOTE to balance the training data (this may take a moment)...')
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_flat, y_train)

print(f'Original training data shape: {X_train_flat.shape}')
print(f'Resampled training data shape: {X_train_resampled.shape}')

# Initialize model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

print('\nTraining Random Forest on SMOTE data...')
rf_model.fit(X_train_resampled, y_train_resampled)
print('Training complete!')


### Evaluate Model Performance


In [ ]:
# Predictions
y_val_pred = rf_model.predict(X_val_flat)
y_val_prob = rf_model.predict_proba(X_val_flat)[:, 1]

print('--- Validation Set Performance ---')
print(classification_report(y_val, y_val_pred))
print(f'ROC-AUC Score: {roc_auc_score(y_val, y_val_prob):.4f}')


### Patient Risk Stratification
Categorizing the probabilities into low, moderate, high, and very high risk levels.

In [ ]:
import pandas as pd

# Create a DataFrame to view the final predictions and risk levels
results_df = pd.DataFrame({
    'True_Label': y_val,
    'Deterioration_Predicted': y_val_pred,
    'Deterioration_Probability': y_val_prob
})

# Define risk categories based on probability thresholds
# You can adjust these thresholds based on clinical requirements
def categorize_risk(prob):
    if prob < 0.25:
        return 'Low Risk'
    elif prob < 0.50:
        return 'Moderate Risk'
    elif prob < 0.75:
        return 'High Risk'
    else:
        return 'Very High Risk'

results_df['Risk_Level'] = results_df['Deterioration_Probability'].apply(categorize_risk)
results_df['Prediction_Text'] = results_df['Deterioration_Predicted'].map({0: 'No', 1: 'Yes'})

print('Sample of Patient Predictions:')
display(results_df[['True_Label', 'Deterioration_Probability', 'Risk_Level', 'Prediction_Text']].head(20))

print('\nRisk Level Distribution in Validation Set:')
print(results_df['Risk_Level'].value_counts())
